# gerbil-data: Feature Engineering Pipeline for Recommender Systems

End-to-end demo using the MovieLens 1M dataset.

In [ ]:
import os, json, subprocess, shutil, pandas as pd

PROJECT_HOME = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = "/tmp/ml-1m"
OUTPUT_DIR = "/tmp/gerbil-output"
JAR = os.path.join(PROJECT_HOME, "target/gerbil-data-1.0.0-jar-with-dependencies.jar")

assert os.path.exists(DATA_DIR), f"Dataset not found at {DATA_DIR}"
assert os.path.exists(JAR), f"JAR not found at {JAR}. Run: mvn clean package -DskipTests"
print(f"Project: {PROJECT_HOME}\nData:    {DATA_DIR}\nJAR:     {JAR}")

## 1. Raw Data

In [ ]:
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.dat"), sep="::",
    names=["UserID","MovieID","Rating","Timestamp"], engine="python", encoding="latin-1")
print(f"Ratings: {ratings.shape[0]:,} rows, {ratings.UserID.nunique():,} users, {ratings.MovieID.nunique():,} movies")
ratings.head()

In [ ]:
users = pd.read_csv(os.path.join(DATA_DIR, "users.dat"), sep="::",
    names=["UserID","Gender","Age","Occupation","ZipCode"], engine="python", encoding="latin-1")
print(f"Users: {users.shape[0]:,} rows")
users.head()

In [ ]:
movies = pd.read_csv(os.path.join(DATA_DIR, "movies.dat"), sep="::",
    names=["MovieID","Title","Genres"], engine="python", encoding="latin-1")
print(f"Movies: {movies.shape[0]:,} rows")
movies.head()

## 2. ETL Pipeline (Spark)

In [ ]:
def spark_run(cls, *extra):
    cmd = ["spark-submit", "--class", cls, JAR, DATA_DIR] + list(extra)
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_HOME)
    ok = r.returncode == 0
    print(f"  {'OK' if ok else 'FAIL'} {cls.split('.')[-1]}")
    if not ok:
        print(f"    {r.stderr[-300:]}")
    return ok

print("Running ETL stages...")
for cls in ["processing.clean.ML1MCleanSample",
            "processing.feature.ML1MMovieStatFeature",
            "processing.feature.ML1MUserMovieRateSequence",
            "processing.join.ML1MJoinSample"]:
    spark_run(cls)

## 3. Joined Feature Table

In [ ]:
join_path = os.path.join(DATA_DIR, "join_sample")
part = sorted(f for f in os.listdir(join_path) if f.endswith('.csv') and not f.startswith('.'))[0]
df = pd.read_csv(os.path.join(join_path, part), sep='\t', nrows=3, escapechar='\\',
    names=["uid","iid","ts","rating","day","user_prof","item_feat","user_beh"])
up = json.loads(df.iloc[0]["user_prof"])
print("User profile:", json.dumps(up, indent=2))
mf = json.loads(df.iloc[0]["item_feat"])
print("\nMovie feature:", json.dumps(mf, indent=2))

## 4. Featurization (Spark)

In [ ]:
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

cmd = ["spark-submit", "--class", "pipeline.ML1MPipeline",
       "--conf", "spark.serializer=org.apache.spark.serializer.JavaSerializer",
       JAR, "--yesterday", "20010101", "--parts", "1",
       "--feature_threshold", "5", "--target_threshold", "5",
       "--sample_ratio", "0.1",
       "--input_dir", DATA_DIR, "--output_dir", OUTPUT_DIR,
       "--output_format", "tfrecord"]
print("Running featurization...")
r = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_HOME)
for line in r.stdout.split(chr(10)):
    if any(kw in line for kw in ["Total:", "Valid:", "pos_map", "target_map"]):
        print(f"  {line}")
if r.returncode != 0:
    print(f"FAILED: {r.stderr[-300:]}")
else:
    print(f"\nOutput: {OUTPUT_DIR}/20010101/")

## 5. TFRecord Output

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "tensorflow", "-q"], capture_output=True)
import tensorflow as tf

tf_dir = os.path.join(OUTPUT_DIR, "20010101", "tfrecord")
files = sorted(f for f in os.listdir(tf_dir) if f.endswith(".tfrecord"))
ds = tf.data.TFRecordDataset(os.path.join(tf_dir, files[0]))
for i, raw in enumerate(ds.take(1)):
    ex = tf.train.Example()
    ex.ParseFromString(raw.numpy())
    print(f"Record {i}: {len(ex.features.feature)} feature fields")
    for name in sorted(ex.features.feature)[:12]:
        feat = ex.features.feature[name]
        kind = feat.WhichOneof('kind').replace('_list', '')
        vals = list(getattr(feat, feat.WhichOneof('kind')).value)[:2]
        print(f"  {name:<30} {kind:<8} {vals}")

## 6. Vocabulary

In [ ]:
with open(os.path.join(OUTPUT_DIR, "20010101", "pos_map.json")) as f:
    vocab = json.load(f)
print(f"Vocabulary entries: {len(vocab)}")
for entry in vocab[:5]:
    print(f"  {entry}")

csv_path = os.path.join(OUTPUT_DIR, "20010101", "pos_map.txt")
if os.path.exists(csv_path):
    dims = pd.read_csv(csv_path, sep="\t")
    print(f"\n{dims.shape[0]} feature fields:")
    print(dims.to_string(index=False))

## Summary

1. Raw data inspection - complete
2. ETL pipeline - complete (raw data -> joined feature table)
3. Featurization - complete (features -> `{name}_raw/_index/_value` triplets)
4. TFRecord output - complete (ready for TensorFlow training)
5. Vocabulary - complete (embedding position maps for offline/online serving)